<a href="https://colab.research.google.com/github/NarendraRaoJami/J_V_Narendra_Rao_Internship_2026_College/blob/main/J_V_Narendra_Rao/DuQuant_BERT_Benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets evaluate accelerate pynvml

In [2]:
import time, gc, warnings, threading
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
import evaluate as ev

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

MODELS = {
    'TinyBERT':   'huawei-noah/TinyBERT_General_4L_312D',
    'DistilBERT': 'distilbert-base-uncased',
    'AlBERT':     'albert-base-v2',
    'MobileBERT': 'google/mobilebert-uncased',
    'BERT-base':  'bert-base-uncased',
}

DATASETS = {
    'SST2': ('stanfordnlp/sst2',  None,   'validation',         'sentence',               'label'),
    'QNLI': ('nyu-mll/glue',      'qnli', 'validation',         ('question','sentence'),   'label'),
    'MNLI': ('nyu-mll/glue',      'mnli', 'validation_matched', ('premise','hypothesis'),  'label'),
    'QQP':  ('nyu-mll/glue',      'qqp',  'validation',         ('question1','question2'), 'label'),
    'RTE':  ('nyu-mll/glue',      'rte',  'validation',         ('sentence1','sentence2'), 'label'),
    # ---ADDED---
    'CoLA': ('nyu-mll/glue',      'cola', 'validation',         'sentence',               'label'),
    'MRPC': ('nyu-mll/glue',      'mrpc', 'validation',         ('sentence1','sentence2'), 'label'),
    'STSB': ('nyu-mll/glue',      'stsb', 'validation',         ('sentence1','sentence2'), 'label'),
    'WNLI': ('nyu-mll/glue',      'wnli', 'validation',         ('sentence1','sentence2'), 'label'),
}

BATCH_SIZE      = 32
MAX_SAMPLES     = 500
MAX_LENGTH      = 128
QUANT_BITS      = 8
POLL_INTERVAL_S = 0.005  # 5 ms polling interval for power sampler


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Device: cuda


In [3]:
# ---------------------------------------------------------------------------
# pynvml calls the NVML C library directly (no process spawn overhead),
# allowing us to poll at ~5 ms intervals and time-average power across the
# entire forward pass instead of taking a single snapshot before it.
# ---------------------------------------------------------------------------
try:
    import pynvml
    pynvml.nvmlInit()
    _nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    NVML_AVAILABLE = True
    print(f'pynvml ready | GPU: {pynvml.nvmlDeviceGetName(_nvml_handle)}')
except Exception as e:
    NVML_AVAILABLE = False
    print(f'pynvml unavailable ({e}) — energy will be reported as 0.0 mJ')


class PowerSampler:
    def __init__(self):
        self._samples = []
        self._running = False
        self._thread  = None

    def _poll(self):
        while self._running:
            if NVML_AVAILABLE:
                try:
                    # nvmlDeviceGetPowerUsage returns milliwatts directly (no conversion needed)
                    self._samples.append(pynvml.nvmlDeviceGetPowerUsage(_nvml_handle))
                except Exception:
                    pass
            time.sleep(POLL_INTERVAL_S)

    def start(self):
        self._samples = []
        self._running = True
        self._thread  = threading.Thread(target=self._poll, daemon=True)
        self._thread.start()

    def stop(self):
        """Stop sampling and return mean power in mW (0.0 if unavailable)."""
        self._running = False
        self._thread.join(timeout=0.5)
        return float(np.mean(self._samples)) if self._samples else 0.0


pynvml ready | GPU: Tesla T4


In [4]:
def get_model(hf_id, num_labels):
    tok = AutoTokenizer.from_pretrained(hf_id)
    model = AutoModelForSequenceClassification.from_pretrained(
        hf_id, num_labels=num_labels, ignore_mismatched_sizes=True
    ).half().to(DEVICE)
    return tok, model


def memory_mb(model, quant_bits=8):
    # Each FP16 parameter = 2 bytes.
    # INT8 uses 1 byte per parameter (quant_bits/16 = 8/16 = 0.5).
    fp16_bytes = sum(p.numel() * 2 for p in model.parameters())
    return round(fp16_bytes * (quant_bits / 16) / 1e6, 2)


def tokenize(tok, texts):
    if isinstance(texts[0], tuple):
        return tok([t[0] for t in texts], [t[1] for t in texts],
                   truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors='pt')
    return tok(texts, truncation=True, padding=True,
               max_length=MAX_LENGTH, return_tensors='pt')


def benchmark(model, tok, ds_cfg):
    path, config, split, text_col, label_col = ds_cfg
    ds = load_dataset(path, config, split=split) if config else load_dataset(path, split=split)
    ds = ds.select(range(min(MAX_SAMPLES, len(ds))))

    model.eval()
    preds, labels, latencies, energies = [], [], [], []
    t_start = time.perf_counter()

    for i in range(0, len(ds), BATCH_SIZE):
        batch = ds[i:i+BATCH_SIZE]
        texts = list(zip(batch[text_col[0]], batch[text_col[1]])) if isinstance(text_col, tuple) else batch[text_col]
        enc = {k: v.to(DEVICE) for k, v in tokenize(tok, texts).items()}

        sampler = PowerSampler()
        sampler.start()

        t0 = time.perf_counter()
        with torch.no_grad():
            out = model(**enc)

        if DEVICE == 'cuda':
            torch.cuda.synchronize()
        elapsed_ms = (time.perf_counter() - t0) * 1000

        avg_power_mw = sampler.stop()

        batch_size_actual = len(batch[label_col])
        energy_mj = (avg_power_mw * elapsed_ms * 1e-3) / batch_size_actual

        latencies.append(elapsed_ms / batch_size_actual)
        energies.append(energy_mj)
        if config == 'stsb':
            preds.extend(out.logits.squeeze(-1).cpu().tolist())
        else:
            preds.extend(out.logits.argmax(-1).cpu().tolist())
        labels.extend(batch[label_col])

    total_time = time.perf_counter() - t_start
    if config == 'stsb':
      # Use Pearson correlation as the metric (standard for STS-B)
      acc = float(np.corrcoef(preds, labels)[0, 1])  # Pearson r
    else:
      acc = ev.load('accuracy').compute(predictions=preds, references=labels)['accuracy']
    lat_ms     = np.mean(latencies)
    throughput = len(ds) / total_time
    energy_mj  = np.mean(energies)
    return round(acc * 100, 2), round(lat_ms, 4), round(throughput, 1), round(energy_mj, 4)


In [5]:
NUM_LABELS = {'SST2': 2, 'QNLI': 2, 'MNLI': 3, 'QQP': 2, 'RTE': 2,
              'CoLA' : 2, 'MRPC' : 2,'STSB' : 1,'WNLI' : 2}
results = []

for ds_name, ds_cfg in DATASETS.items():
    for model_name, hf_id in MODELS.items():
        print(f'{ds_name} | {model_name} ... ', end='', flush=True)
        try:
            tok, model = get_model(hf_id, NUM_LABELS[ds_name])
            mem = memory_mb(model, QUANT_BITS)
            acc, lat, tp, energy = benchmark(model, tok, ds_cfg)
            results.append({
                'Dataset': ds_name, 'Model': model_name,
                'Memory (MB)': mem, 'Latency (ms)': lat,
                'Accuracy (%)': acc, 'Bits': QUANT_BITS,
                'Energy (mJ)': energy, 'Throughput (sps)': tp
            })
            print(f'Acc={acc:.1f}%  Lat={lat:.3f}ms  Mem={mem}MB  Energy={energy:.4f}mJ  TP={tp:.0f}sps')
        except Exception as e:
            print(f'ERROR: {e}')
        finally:
            try:
                del model, tok
                gc.collect()
                torch.cuda.empty_cache()
            except:
                pass


SST2 | TinyBERT ... 

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     | 
-------------------------------------------+------------+-
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not o

Acc=47.4%  Lat=0.758ms  Mem=14.35MB  Energy=22.6867mJ  TP=985sps
SST2 | DistilBERT ... 

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

MobileBertForSequenceClassification LOAD REPORT from: google/mobilebert-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
mobilebert.embeddings.position_ids         | UNEXPECTED | 
cls.predictions.dense.weight               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignore

Acc=56.3%  Lat=7.553ms  Mem=24.58MB  Energy=242.4393mJ  TP=188sps
WNLI | BERT-base ... 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Acc=56.3%  Lat=1.820ms  Mem=109.48MB  Energy=85.2795mJ  TP=428sps


In [6]:
df = pd.DataFrame(results).set_index(['Dataset', 'Model'])
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)
pd.set_option('display.float_format', '{:.4f}'.format)

for ds in df.index.get_level_values('Dataset').unique():
    print(f'\n========== {ds} ==========')
    print(df.loc[ds].to_string())
print()



========== SST2 ==========
            Memory (MB)  Latency (ms)  Accuracy (%)  Bits  Energy (mJ)  Throughput (sps)
Model                                                                                   
TinyBERT        14.3500        0.7577       47.4000     8      22.6867          985.2000
DistilBERT      66.9600        0.4662       53.0000     8      16.4664         1036.4000
AlBERT          11.6900        0.9413       53.6000     8      46.6993          776.0000
MobileBERT      24.5800        4.8157       47.0000     8     144.0952          179.6000
BERT-base      109.4800        0.9557       45.8000     8      36.2357          565.1000

========== QNLI ==========
            Memory (MB)  Latency (ms)  Accuracy (%)  Bits  Energy (mJ)  Throughput (sps)
Model                                                                                   
TinyBERT        14.3500        0.4356       58.8000     8      16.9474         1151.2000
DistilBERT      66.9600        0.7214       45.2000   